In [ ]:
import torch
import sys
from pathlib import Path
import torch, torch.nn as nn
import torchvision.models as models

# Add the project root (parent of NoteBooks) to Python's path
project_root = Path.cwd().parent  # if notebook is in NoteBooks/
sys.path.insert(0, str(project_root))

from src.training.train_cnn import create_cnn_model
from src.models.mlp import create_mlp_model # YIPEE
from src.data.loader import *
from src.data.preprocessing import * # need thsi for just a sec. 

In [2]:
# constants. 
projections_path = "../IU-xray/indiana_projections.csv"
reports_path = "../IU-xray\indiana_reports.csv"
image_dir = '../IU-xray/images/images_normalized'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = Path("../models")


In [1]:
modes = ["abnormal", "pneumonia"]
for mode in modes:
    create_train_test_split(projections_path, reports_path, mode)

NameError: name 'create_train_test_split' is not defined

In [ ]:
modes = ["abnormal", "pneumonia"]

for mode in modes:
    df = build_master_df_rich(projections_path, reports_path, mode) # get EVERYTHING. 
    # subset = True # this is for prototyping purposes. Just to make sure stuff doesn't crash, not that it works well per se. 
    # if subset == True: 
    #     df = df.sample(n=200, random_state=42)
    # this should do the trick
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

    # 1. Train MLP
    print("Training MLP on unified split")
    text_amount = "medium" # use the medium amount of text data
    model_type = "single" # not ensemble, single mode. 
    # returns a state dict, not the model
    MLP_model_state_dict, errors_train, errors_test, losses_train, losses_test = create_mlp_model(df, text_amount, model_type)

    # 2. Train CNN
    print("training finetuned CNN's sire ")
    num_epochs = 25
    CNN_model_state_dict, history = create_cnn_model(df, image_dir, device, mode, num_epochs)

    # save the models...
    MLP_filename = f"best_mlp_{mode}_{text_amount}.pth"
    MLP_full_path = save_path / MLP_filename
    torch.save(MLP_model_state_dict, MLP_full_path)


    CNN_filename = f"best_cnn_densenet_{mode}.pth"
    CNN_full_path = save_path / CNN_filename
    torch.save(CNN_model_state_dict, CNN_full_path)



This is the projections path  ../IU-xray/indiana_projections.csv
Abnormal mode: 2472 positives / 3851 total (64.2%)
Training MLP on unified split
starting the model creation process 
here are the amx featuerts  1000
Training MLP
************************************
  t=0.50 | Sensitivity: 74.7% | Specificity: 71.6% | FN: 123 | FP: 79
  t=0.30 | Sensitivity: 81.1% | Specificity: 59.4% | FN: 92 | FP: 113
  t=0.20 | Sensitivity: 87.7% | Specificity: 47.1% | FN: 60 | FP: 147
  t=0.10 | Sensitivity: 93.6% | Specificity: 33.1% | FN: 31 | FP: 186
  t=0.05 | Sensitivity: 97.7% | Specificity: 20.1% | FN: 11 | FP: 222
Test Accuracy : 73.6%
Test AUC-ROC  : 0.8295

Confusion Matrix:
  True Negatives  (correctly said normal)   : 56
  False Positives (incorrectly flagged)      : 222
  False Negatives (missed abnormal cases)   : 11  ← this is the dangerous one
  True Positives  (correctly caught)         : 475

Sensitivity (recall on positives): 97.7%
Specificity (recall on negatives): 20.1%
training